# Week 05 Assignment: Build a Tiny 2-Block Transformer from Scratch in Pure NumPy
## *From Context Gathering to Deep Non-Linear Decision Machinery*

**Course**: AI Learning Lab — Advanced Transformer Architecture Series  
**Domain Focus**: Healthcare GPO (Group Purchasing Organization) & USAID Supply Chain Audit Lifecycles  
**Implementation Scope**: Pure NumPy (Zero PyTorch `nn.Transformer` or High-Level Deep Learning Frameworks)  

---

## Section 1: Executive Summary & Architecture Blueprint

### 1. What We Are Building
In Week 4, we built Multi-Head Attention—the mechanism that decides **where to look** and **what context to gather**. In Week 5, we complete the full Transformer block by building the machinery that decides **what to do with what it found**:
1. **Layer Normalization (LayerNorm)**: Standardizes activation vectors to mean 0 and variance 1 across feature dimensions.
2. **Position-Wise Feed-Forward Network (FFN)**: Expands hidden dimensions by 4x (d_model -> 4*d_model -> d_model) with ReLU non-linearity, acting as an associative key-value memory bank.
3. **Residual Connections (Skip Highways)**: Adds unmodified input X directly to sub-layer outputs (X + f(X)), creating a +1.0 derivative bypass that eliminates vanishing gradients.
4. **Stacked Block Depth (N=2)**: Passes representations sequentially through Block 1 and Block 2 to allow hierarchical representation evolution.

We benchmark **3 Architectural Baselines** on an expanded **Healthcare GPO Supply Chain Dataset (1,000 synthetic log sequences across 15 procurement tokens)** under identical seed `42` initialization:
- **Model A**: Embedding -> Linear Predictor
- **Model B**: Embedding -> Multi-Head Attention -> Linear Predictor
- **Model C**: Embedding -> 2-Block Transformer -> Linear Predictor

### 2. Complete Architecture Blueprint (Pre-LN 2-Block Transformer)

```
               Input Tokens (Batch B x Seq_Len T)
                              |
               [ Token Embedding + Sinusoidal PE ]
                              |
                    Tensor X_0 (B x T x d_model)
                              |
   ======================= BLOCK 1 =======================
   |  X_0 ---+-------------------------------┐           |
   |         |                               |           |
   |         v                               | (Residual)|
   |   [ Multi-Head Attention (H=4) ]        |           |
   |         |                               |           |
   |         v                               |           |
   |       MHA(X_0)                          |           |
   |         |                               |           |
   |         └───────────────> (+) <─────────┘           |
   |                            |                        |
   |                    [ LayerNorm 1 ]                  |
   |                            |                        |
   |                            v                        |
   |                  SubLayer_1 (B x T x d_model)       |
   |                            |                        |
   |  SubLayer_1 --+-----------------------------┐       |
   |               |                             |       |
   |               v                             | (Res) |
   |         [ Position-Wise FFN (4xd_model) ]   |       |
   |               |                             |       |
   |               v                             |       |
   |             FFN(SubLayer_1)                 |       |
   |               |                             |       |
   |               └─────────> (+) <─────────────┘       |
   |                            |                        |
   |                    [ LayerNorm 2 ]                  |
   =============================|=========================
                                v
                    Tensor X_1 (B x T x d_model)
                                |
   ======================= BLOCK 2 =======================
   |     (Identical Sub-layer Architecture as Block 1)   |
   =============================|=========================
                                v
                    Tensor X_2 (B x T x d_model)
                                |
                [ Vocabulary Projection W_vocab ]
                                |
                Next-Token Logits (B x T x Vocab_Size)
```


## Section 2: Healthcare GPO Supply Chain Dataset Generation

### 1. What We Are Doing
We build an expanded stochastic GPO supply chain workflow generator (adapted from USAID Health Commodity & Hospital Procurement schemas on Kaggle) that synthesizes **1,000 realistic log sequences** across 15 domain tokens (`Requisition`, `PO_Created`, `Fulfillment`, `Shipment`, `Warehouse_Receipt`, `Inventory_Check`, `Invoice_Matching`, `Rebate_Claim`, `Audit_NCR`, `Receive`, `Restock`, `Forecast`, `Order`, `Scenario`, `Contract`).

### 2. Implementation Code


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure 100% strict reproducibility across all training runs
np.random.seed(42)

# Expanded USAID / Healthcare GPO Transition Graph
gpo_transition_rules = {
    'Requisition': ['PO_Created', 'Contract'],
    'PO_Created': ['Fulfillment', 'Shipment'],
    'Fulfillment': ['Shipment'],
    'Shipment': ['Warehouse_Receipt', 'Receive'],
    'Warehouse_Receipt': ['Inventory_Check', 'Restock'],
    'Receive': ['Restock', 'Inventory_Check'],
    'Restock': ['Inventory_Check', 'Inventory'],
    'Inventory_Check': ['Invoice_Matching', 'Forecast'],
    'Inventory': ['Forecast'],
    'Invoice_Matching': ['Rebate_Claim', 'Audit_NCR'],
    'Rebate_Claim': ['Audit_NCR', 'Forecast'],
    'Audit_NCR': ['Scenario', 'Contract'],
    'Forecast': ['Order', 'Scenario'],
    'Order': ['Shipment', 'Fulfillment'],
    'Contract': ['Requisition', 'PO_Created'],
    'Scenario': ['Contract', 'Order']
}

def generate_gpo_dataset(num_samples=1000, seq_len=5):
    start_nodes = list(gpo_transition_rules.keys())
    generated_seqs = []
    
    for _ in range(num_samples):
        curr_node = np.random.choice(start_nodes)
        seq = [curr_node]
        for _ in range(seq_len - 1):
            possible_next = gpo_transition_rules.get(curr_node, start_nodes)
            curr_node = np.random.choice(possible_next)
            seq.append(curr_node)
        generated_seqs.append(seq)
    return generated_seqs

# Generate 1,000 GPO log sequences of length 5
sequences = generate_gpo_dataset(num_samples=1000, seq_len=5)

# Vocabulary Construction
unique_tokens = sorted(list(set(token for seq in sequences for token in seq)))
vocab = {token: idx for idx, token in enumerate(unique_tokens)}
id_to_vocab = {idx: token for token, idx in vocab.items()}
vocab_size = len(vocab)

print("=== GENERATED HEALTHCARE GPO DATASET METADATA ===")
print(f"Total Generated Sequences : {len(sequences)}")
print(f"Vocabulary Size           : {vocab_size} GPO tokens")
print(f"Vocabulary List           : {unique_tokens}\n")

# Map sequences to token IDs
sequences_ids = [[vocab[token] for token in seq] for seq in sequences]
inputs = np.array([seq[:-1] for seq in sequences_ids])  # (1000, 4)
targets = np.array([seq[1:] for seq in sequences_ids]) # (1000, 4)

num_samples, seq_length = inputs.shape
print(f"Inputs Matrix Shape  (B x T): {inputs.shape}")
print(f"Targets Matrix Shape (B x T): {targets.shape}\n")


## Section 3: Token Embeddings & Sinusoidal Positional Encoding

### 1. What We Are Doing
We implement dense token embeddings (d_model = 16) combined with sinusoidal positional encodings PE.


In [ ]:
d_model = 16

def generate_sinusoidal_pe(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            div_term = np.exp(i * -np.log(10000.0) / d_model)
            PE[pos, i] = np.sin(pos * div_term)
            if i + 1 < d_model:
                PE[pos, i + 1] = np.cos(pos * div_term)
    return PE

PE = generate_sinusoidal_pe(seq_length, d_model)

def get_embedded_input(inputs, W_embed, PE):
    token_embeds = W_embed[inputs] # (B, T, d_model)
    X = token_embeds + PE         # Broadcasts PE over batch
    return X


## Section 4: Pure NumPy Layer Normalization (Forward & Backward)

### 1. What We Are Doing
We write Layer Normalization from scratch in pure NumPy, standardizing each token feature vector across d_model to have mean 0 and variance 1, with learnable gain gamma and bias beta.


In [ ]:
def layer_norm_forward(x, gamma, beta, eps=1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    x_hat = (x - mean) / np.sqrt(var + eps)
    out = gamma * x_hat + beta
    cache = (x, x_hat, mean, var, gamma, beta, eps)
    return out, cache

def layer_norm_backward(dout, cache):
    x, x_hat, mean, var, gamma, beta, eps = cache
    N = x.shape[-1]
    
    dgamma = np.sum(dout * x_hat, axis=(0, 1))
    dbeta = np.sum(dout, axis=(0, 1))
    
    dx_hat = dout * gamma
    std_inv = 1.0 / np.sqrt(var + eps)
    
    dx = (1.0 / N) * std_inv * (
        N * dx_hat -
        np.sum(dx_hat, axis=-1, keepdims=True) -
        x_hat * np.sum(dx_hat * x_hat, axis=-1, keepdims=True)
    )
    return dx, dgamma, dbeta


## Section 5: Position-Wise Feed-Forward Network (FFN)

### 1. What We Are Doing
We implement the 2-layer FFN with 4x hidden dimension expansion (d_ff = 64) and ReLU non-linearity.


In [ ]:
d_ff = 4 * d_model # 4 * 16 = 64

def relu(x):
    return np.maximum(0, x)

def relu_backward(dout, x):
    dx = dout.copy()
    dx[x <= 0] = 0
    return dx

def ffn_forward(x, W1, b1, W2, b2):
    h_pre = x @ W1 + b1 # (B, T, 64)
    h = relu(h_pre)     # (B, T, 64)
    out = h @ W2 + b2   # (B, T, 16)
    cache = (x, h_pre, h, W1, b1, W2, b2)
    return out, cache

def ffn_backward(dout, cache):
    x, h_pre, h, W1, b1, W2, b2 = cache
    B, T, D = x.shape
    
    dW2 = h.reshape(-1, d_ff).T @ dout.reshape(-1, D)
    db2 = np.sum(dout, axis=(0, 1))
    dh = dout @ W2.T
    
    dh_pre = relu_backward(dh, h_pre)
    dW1 = x.reshape(-1, D).T @ dh_pre.reshape(-1, d_ff)
    db1 = np.sum(dh_pre, axis=(0, 1))
    dx = dh_pre @ W1.T
    
    return dx, dW1, db1, dW2, db2


## Section 6: Multi-Head Attention & Causal Mask Helper Functions

### 1. What We Are Doing
We reuse our 4-head causal attention implementation (H=4, d_head=4).


In [ ]:
num_heads = 4
d_head = d_model // num_heads # 16 // 4 = 4

def create_causal_mask(seq_len):
    return np.triu(np.full((seq_len, seq_len), -np.inf), k=1)

causal_mask = create_causal_mask(seq_length)

def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def forward_multi_head(X, mh_W_Q, mh_W_K, mh_W_V, mh_W_O):
    head_outs = []
    A_list = []
    head_caches = []
    
    for i in range(num_heads):
        Q_i = X @ mh_W_Q[i]
        K_i = X @ mh_W_K[i]
        V_i = X @ mh_W_V[i]
        
        scores_i = (Q_i @ K_i.transpose(0, 2, 1)) / np.sqrt(d_head)
        masked_scores_i = scores_i + causal_mask
        A_i = softmax(masked_scores_i, axis=-1)
        A_list.append(A_i)
        
        head_out_i = A_i @ V_i
        head_outs.append(head_out_i)
        head_caches.append((Q_i, K_i, V_i, scores_i, masked_scores_i, A_i, head_out_i))
        
    concat_out = np.concatenate(head_outs, axis=-1)
    out = concat_out @ mh_W_O
    cache = (X, head_caches, concat_out, out)
    return out, A_list, cache

def backward_multi_head(d_out, cache, mh_W_Q, mh_W_K, mh_W_V, mh_W_O):
    X, head_caches, concat_out, out = cache
    B, T, _ = X.shape
    
    d_W_O = concat_out.reshape(-1, d_model).T @ d_out.reshape(-1, d_model)
    d_concat_out = d_out @ mh_W_O.T
    d_head_outs = np.split(d_concat_out, num_heads, axis=-1)
    
    d_mh_W_Q, d_mh_W_K, d_mh_W_V = [], [], []
    d_X_total = np.zeros_like(X)
    
    for i in range(num_heads):
        Q_i, K_i, V_i, scores_i, masked_scores_i, A_i, head_out_i = head_caches[i]
        d_head_out_i = d_head_outs[i]
        
        d_V_i = A_i.transpose(0, 2, 1) @ d_head_out_i
        d_A_i = d_head_out_i @ V_i.transpose(0, 2, 1)
        d_masked_scores_i = A_i * (d_A_i - np.sum(d_A_i * A_i, axis=-1, keepdims=True))
        d_scores_i = np.where(causal_mask == 0, d_masked_scores_i, 0.0)
        
        d_Q_i = (d_scores_i @ K_i) / np.sqrt(d_head)
        d_K_i = (d_scores_i.transpose(0, 2, 1) @ Q_i) / np.sqrt(d_head)
        
        d_mh_W_Q.append(X.reshape(-1, d_model).T @ d_Q_i.reshape(-1, d_head))
        d_mh_W_K.append(X.reshape(-1, d_model).T @ d_K_i.reshape(-1, d_head))
        d_mh_W_V.append(X.reshape(-1, d_model).T @ d_V_i.reshape(-1, d_head))
        d_X_total += d_Q_i @ mh_W_Q[i].T + d_K_i @ mh_W_K[i].T + d_V_i @ mh_W_V[i].T
        
    return d_X_total, d_mh_W_Q, d_mh_W_K, d_mh_W_V, d_W_O


## Section 7: Assembly & Training of 3 Architectural Baselines

### 1. What We Are Doing
We train 3 distinct model architectures for 1,500 epochs with learning rate eta = 0.15 under identical random seed `42` initialization:
- **Model A**: Embedding -> Linear Predictor
- **Model B**: Embedding -> Multi-Head Attention -> Linear Predictor
- **Model C**: Embedding -> 2-Block Transformer -> Linear Predictor


In [ ]:
def cross_entropy_loss(logits, targets):
    B, T, V = logits.shape
    probs = softmax(logits, axis=-1)
    probs_flat = probs.reshape(B * T, V)
    targets_flat = targets.reshape(-1)
    
    correct_probs = probs_flat[np.arange(B * T), targets_flat]
    loss = -np.mean(np.log(correct_probs + 1e-9))
    
    d_logits = probs_flat.copy()
    d_logits[np.arange(B * T), targets_flat] -= 1.0
    d_logits = (d_logits / (B * T)).reshape(B, T, V)
    return loss, d_logits

epochs = 1500
lr = 0.15

# Model A Weights (Linear Predictor)
ma_W_embed = np.random.randn(vocab_size, d_model) * 0.1
ma_W_vocab = np.random.randn(d_model, vocab_size) * 0.1

# Model B Weights (Single Layer MHA)
mb_W_embed = np.random.randn(vocab_size, d_model) * 0.1
mb_mh_W_Q = [np.random.randn(d_model, d_head) * 0.1 for _ in range(num_heads)]
mb_mh_W_K = [np.random.randn(d_model, d_head) * 0.1 for _ in range(num_heads)]
mb_mh_W_V = [np.random.randn(d_model, d_head) * 0.1 for _ in range(num_heads)]
mb_mh_W_O = np.random.randn(d_model, d_model) * 0.1
mb_W_vocab = np.random.randn(d_model, vocab_size) * 0.1

# Model C Weights (2-Block Transformer)
mc_W_embed = np.random.randn(vocab_size, d_model) * 0.1
mc_blocks = []
for b in range(2):
    mc_blocks.append({
        'W_Q': [np.random.randn(d_model, d_head) * 0.1 for _ in range(num_heads)],
        'W_K': [np.random.randn(d_model, d_head) * 0.1 for _ in range(num_heads)],
        'W_V': [np.random.randn(d_model, d_head) * 0.1 for _ in range(num_heads)],
        'W_O': np.random.randn(d_model, d_model) * 0.1,
        'g1': np.ones(d_model), 'b1': np.zeros(d_model),
        'g2': np.ones(d_model), 'b2': np.zeros(d_model),
        'W1': np.random.randn(d_model, d_ff) * 0.1, 'b1_ff': np.zeros(d_ff),
        'W2': np.random.randn(d_ff, d_model) * 0.1, 'b2_ff': np.zeros(d_model)
    })
mc_W_vocab = np.random.randn(d_model, vocab_size) * 0.1

loss_A, loss_B, loss_C = [], [], []

for epoch in range(epochs):
    # --- Model A Step ---
    xa_X = get_embedded_input(inputs, ma_W_embed, PE)
    xa_logits = xa_X @ ma_W_vocab
    la, d_logits_a = cross_entropy_loss(xa_logits, targets)
    d_xa_X = d_logits_a @ ma_W_vocab.T
    d_ma_W_vocab = xa_X.reshape(-1, d_model).T @ d_logits_a.reshape(-1, vocab_size)
    ma_W_vocab -= lr * d_ma_W_vocab
    np.add.at(ma_W_embed, inputs, -lr * d_xa_X)
    loss_A.append(la)
    
    # --- Model B Step ---
    xb_X = get_embedded_input(inputs, mb_W_embed, PE)
    xb_mha, _, xb_cache = forward_multi_head(xb_X, mb_mh_W_Q, mb_mh_W_K, mb_mh_W_V, mb_mh_W_O)
    xb_logits = xb_mha @ mb_W_vocab
    lb, d_logits_b = cross_entropy_loss(xb_logits, targets)
    d_xb_mha = d_logits_b @ mb_W_vocab.T
    d_mb_W_vocab = xb_mha.reshape(-1, d_model).T @ d_logits_b.reshape(-1, vocab_size)
    d_xb_X, d_mb_WQ, d_mb_WK, d_mb_WV, d_mb_WO = backward_multi_head(d_xb_mha, xb_cache, mb_mh_W_Q, mb_mh_W_K, mb_mh_W_V, mb_mh_W_O)
    mb_W_vocab -= lr * d_mb_W_vocab
    mb_mh_W_O -= lr * d_mb_WO
    for i in range(num_heads):
        mb_mh_W_Q[i] -= lr * d_mb_WQ[i]
        mb_mh_W_K[i] -= lr * d_mb_WK[i]
        mb_mh_W_V[i] -= lr * d_mb_WV[i]
    np.add.at(mb_W_embed, inputs, -lr * d_xb_X)
    loss_B.append(lb)
    
    # --- Model C Step (2-Block Transformer Forward Pass) ---
    xc_X0 = get_embedded_input(inputs, mc_W_embed, PE)
    
    # Block 1 Pre-LN Forward
    n1_b1, cache_n1_b1 = layer_norm_forward(xc_X0, mc_blocks[0]['g1'], mc_blocks[0]['b1'])
    mha1, A1_list, cache_mha1 = forward_multi_head(n1_b1, mc_blocks[0]['W_Q'], mc_blocks[0]['W_K'], mc_blocks[0]['W_V'], mc_blocks[0]['W_O'])
    x1_res = xc_X0 + mha1
    
    n2_b1, cache_n2_b1 = layer_norm_forward(x1_res, mc_blocks[0]['g2'], mc_blocks[0]['b2'])
    ffn1, cache_ffn1 = ffn_forward(n2_b1, mc_blocks[0]['W1'], mc_blocks[0]['b1_ff'], mc_blocks[0]['W2'], mc_blocks[0]['b2_ff'])
    xc_X1 = x1_res + ffn1
    
    # Block 2 Pre-LN Forward
    n1_b2, cache_n1_b2 = layer_norm_forward(xc_X1, mc_blocks[1]['g1'], mc_blocks[1]['b1'])
    mha2, A2_list, cache_mha2 = forward_multi_head(n1_b2, mc_blocks[1]['W_Q'], mc_blocks[1]['W_K'], mc_blocks[1]['W_V'], mc_blocks[1]['W_O'])
    x2_res = xc_X1 + mha2
    
    n2_b2, cache_n2_b2 = layer_norm_forward(x2_res, mc_blocks[1]['g2'], mc_blocks[1]['b2'])
    ffn2, cache_ffn2 = ffn_forward(n2_b2, mc_blocks[1]['W1'], mc_blocks[1]['b1_ff'], mc_blocks[1]['W2'], mc_blocks[1]['b2_ff'])
    xc_X2 = x2_res + ffn2
    
    xc_logits = xc_X2 @ mc_W_vocab
    lc, d_logits_c = cross_entropy_loss(xc_logits, targets)
    loss_C.append(lc)
    
    # --- Model C Backward Pass ---
    d_mc_W_vocab = xc_X2.reshape(-1, d_model).T @ d_logits_c.reshape(-1, vocab_size)
    d_xc_X2 = d_logits_c @ mc_W_vocab.T
    
    # Block 2 Backward
    d_x2_res = d_xc_X2.copy()
    d_ffn2 = d_xc_X2.copy()
    d_n2_b2, dW2_b2, db2_b2, dW1_b2, db1_b2 = ffn_backward(d_ffn2, cache_ffn2)
    d_x2_res_norm, dg2_b2, db2_norm2 = layer_norm_backward(d_n2_b2, cache_n2_b2)
    d_x2_total = d_x2_res + d_x2_res_norm
    
    d_xc_X1 = d_x2_total.copy()
    d_mha2 = d_x2_total.copy()
    d_n1_b2, dWQ2, dWK2, dWV2, dWO2 = backward_multi_head(d_mha2, cache_mha2, mc_blocks[1]['W_Q'], mc_blocks[1]['W_K'], mc_blocks[1]['W_V'], mc_blocks[1]['W_O'])
    d_xc_X1_norm, dg1_b2, db1_norm2 = layer_norm_backward(d_n1_b2, cache_n1_b2)
    d_xc_X1_total = d_xc_X1 + d_xc_X1_norm
    
    # Block 1 Backward
    d_x1_res = d_xc_X1_total.copy()
    d_ffn1 = d_xc_X1_total.copy()
    d_n2_b1, dW2_b1, db2_b1, dW1_b1, db1_b1 = ffn_backward(d_ffn1, cache_ffn1)
    d_x1_res_norm, dg2_b1, db2_norm1 = layer_norm_backward(d_n2_b1, cache_n2_b1)
    d_x1_total = d_x1_res + d_x1_res_norm
    
    d_xc_X0 = d_x1_total.copy()
    d_mha1 = d_x1_total.copy()
    d_n1_b1, dWQ1, dWK1, dWV1, dWO1 = backward_multi_head(d_mha1, cache_mha1, mc_blocks[0]['W_Q'], mc_blocks[0]['W_K'], mc_blocks[0]['W_V'], mc_blocks[0]['W_O'])
    d_xc_X0_norm, dg1_b1, db1_norm1 = layer_norm_backward(d_n1_b1, cache_n1_b1)
    d_xc_X0_total = d_xc_X0 + d_xc_X0_norm
    
    # Update Model C Weights
    mc_W_vocab -= lr * d_mc_W_vocab
    for b_idx, grads in enumerate([
        (dWQ1, dWK1, dWV1, dWO1, dg1_b1, db1_norm1, dg2_b1, db2_norm1, dW1_b1, db1_b1, dW2_b1, db2_b1),
        (dWQ2, dWK2, dWV2, dWO2, dg1_b2, db1_norm2, dg2_b2, db2_norm2, dW1_b2, db1_b2, dW2_b2, db2_b2)
    ]):
        for i in range(num_heads):
            mc_blocks[b_idx]['W_Q'][i] -= lr * grads[0][i]
            mc_blocks[b_idx]['W_K'][i] -= lr * grads[1][i]
            mc_blocks[b_idx]['W_V'][i] -= lr * grads[2][i]
        mc_blocks[b_idx]['W_O'] -= lr * grads[3]
        mc_blocks[b_idx]['g1'] -= lr * grads[4]
        mc_blocks[b_idx]['b1'] -= lr * grads[5]
        mc_blocks[b_idx]['g2'] -= lr * grads[6]
        mc_blocks[b_idx]['b2'] -= lr * grads[7]
        mc_blocks[b_idx]['W1'] -= lr * grads[8]
        mc_blocks[b_idx]['b1_ff'] -= lr * grads[9]
        mc_blocks[b_idx]['W2'] -= lr * grads[10]
        mc_blocks[b_idx]['b2_ff'] -= lr * grads[11]
        
    np.add.at(mc_W_embed, inputs, -lr * d_xc_X0_total)


## Section 8: Loss Convergence & Generalization Plotting

### 1. What We Are Doing
We plot the loss curves of Model A, Model B, and Model C side-by-side to evaluate convergence speed and generalizability.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(loss_A, label="Model A: Embedding -> Linear", color="#e74c3c", linestyle="--", linewidth=2)
plt.plot(loss_B, label="Model B: Embedding -> MHA -> Linear", color="#f39c12", linewidth=2)
plt.plot(loss_C, label="Model C: 2-Block Transformer (MHA + FFN + LN + Res)", color="#2ecc71", linewidth=2.5)
plt.title("3-Model Generalization Benchmark on 1,000 Healthcare GPO Log Sequences", fontsize=13, fontweight="bold")
plt.xlabel("Epochs", fontsize=11)
plt.ylabel("Cross-Entropy Loss", fontsize=11)
plt.legend(fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Final Model A Loss (Linear)           : {loss_A[-1]:.5f}")
print(f"Final Model B Loss (Single MHA)        : {loss_B[-1]:.5f}")
print(f"Final Model C Loss (2-Block Transformer): {loss_C[-1]:.5f}")


## Section 9: Next-Token Accuracy & Prediction Evaluation

### 1. What We Are Doing
We compute top-1 next-token prediction accuracy across all 1,000 healthcare GPO sequences.


In [ ]:
def evaluate_accuracy(logits, targets, name):
    probs = softmax(logits, axis=-1)
    preds = np.argmax(probs, axis=-1)
    acc = (np.sum(preds == targets) / targets.size) * 100.0
    print(f"{name:<35} | Accuracy: {acc:6.2f}%")
    return acc

print("=== 3-MODEL NEXT-TOKEN ACCURACY BENCHMARK ===")
evaluate_accuracy(xa_logits, targets, "Model A (Linear Predictor)")
evaluate_accuracy(xb_logits, targets, "Model B (Single MHA)")
evaluate_accuracy(xc_logits, targets, "Model C (2-Block Transformer)")


## Section 10: Component-by-Component Empirical Ablation Study

### 1. What We Are Doing
We systematically ablate key components from the 2-Block Transformer (Model C):
1. **No FFN**: Replace Feed-Forward Network with identity pass.
2. **No LayerNorm**: Remove normalization steps.
3. **Single Block vs 2 Blocks**: Compare 1-block vs 2-block performance.


In [ ]:
# Ablation 1: 1-Block Transformer Output (X1)
xc_logits_1block = xc_X1 @ mc_W_vocab
ab_acc_1block = (np.sum(np.argmax(softmax(xc_logits_1block, axis=-1), axis=-1) == targets) / targets.size) * 100.0

# Ablation 2: No FFN (Bypassing FFN sub-layer in Block 2)
xc_logits_no_ffn = x2_res @ mc_W_vocab
ab_acc_no_ffn = (np.sum(np.argmax(softmax(xc_logits_no_ffn, axis=-1), axis=-1) == targets) / targets.size) * 100.0

print("=== COMPONENT-BY-COMPONENT ABLATION RESULTS ===")
print(f"Full 2-Block Transformer Accuracy : {(np.sum(np.argmax(softmax(xc_logits, axis=-1), axis=-1) == targets)/targets.size)*100:.2f}%")
print(f"Single Block Only (No Block 2)   : {ab_acc_1block:.2f}%")
print(f"No FFN Sub-layer (MHA Only)      : {ab_acc_no_ffn:.2f}%")


## Section 11: Layer Representation Evolution & Heatmaps

### 1. What We Are Doing
We plot attention heatmaps of Block 1 vs Block 2 side-by-side to observe how context representations evolve from local to global focus.


In [ ]:
sample_idx = 0
seq_labels = [id_to_vocab[idx] for idx in inputs[sample_idx]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(A1_list[0][sample_idx], annot=True, cmap="Blues", fmt=".2f", xticklabels=seq_labels, yticklabels=seq_labels, ax=axes[0])
axes[0].set_title("Block 1 (Head 1) Attention Matrix", fontweight="bold")

sns.heatmap(A2_list[0][sample_idx], annot=True, cmap="Greens", fmt=".2f", xticklabels=seq_labels, yticklabels=seq_labels, ax=axes[1])
axes[1].set_title("Block 2 (Head 1) Attention Matrix", fontweight="bold")

plt.suptitle("Hierarchical Layer Representation Evolution (Block 1 vs Block 2)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## Section 12: Assignment Synthesis, Flashcards, & Quick Revision Guide

| Component | Key Function | What It Buys Us |
| :--- | :--- | :--- |
| **Multi-Head Attention** | Context Gathering | Decides **where to look** across multiple representation subspaces. |
| **Feed-Forward Network** | Feature Processing | Decides **what to do** with gathered context (4x associative memory expansion). |
| **Residual Connection** | Skip Highway | +1.0 derivative bypass that eliminates vanishing gradients in deep networks. |
| **Layer Normalization** | Activation Leveler | Rescales feature vectors to mean 0 and variance 1, preventing numeric explosion. |
| **Stacked Block Depth (N=2)** | Representation Evolution | Enables hierarchical feature abstraction from local syntax to global context. |
